In [ ]:
cd Q:\sachuriga\Sachuriga_Python\quattrocolo-nwb4fp\src

In [ ]:
from neurochat.nc_data import NData
from neurochat.nc_spike import NSpike
from neurochat.nc_spatial import NSpatial
import neurochat.nc_plot as nc_plot
from neurochat.nc_lfp import NLfp
import matplotlib.pyplot as plt
import numpy as np
from pynwb import NWBHDF5IO
import matplotlib.pyplot as plt
import numpy as np
import math
import pynapple as nap
import numpy as np
from scipy import signal
import matplotlib.pyplot as plt
import numpy as np
from sklearn.preprocessing import normalize
import os
os.chdir(r"/Users/sachuriga/Desktop/code/nwb4fp/src")
import sys
import nwb4fp.analyses.maps as mapp
from nwb4fp.analyses.examples.tracking_plot import plot_ratemap,plot_path,plot_ratemap_ax
from nwb4fp.analyses.fields import separate_fields_by_laplace, separate_fields_by_dilation,find_peaks,separate_fields_by_laplace_of_gaussian,calculate_field_centers,distance_to_edge_function, remove_fields_by_area, map_pass_to_unit_circle,which_field,compute_crossings
from elephant.statistics import time_histogram, instantaneous_rate
from nwb4fp.analyses import maps
from nwb4fp.analyses.data import pos2speed,speed_filtered_spikes,load_speed_fromNWB,load_units_fromNWB,find_run_indices
from nwb4fp.data.helpers import unit_location_ch
from scipy.ndimage import gaussian_filter
from nwb4fp.preprocess.down_sample_lfp import down_sample_lfp_test
import ast
import numpy as np
from scipy import signal
from scipy.ndimage import gaussian_filter1d
import matplotlib.pyplot as plt
import os
import pandas as pd

In [ ]:
file_path = r"/Users/sachuriga/Desktop/code/nwb4fp/src/ASSY-236-F.prb"
# Read the file and parse the dictionary
local_vars = {'np': np}
with open(file_path, 'r') as file:
    exec(file.read(), local_vars)  # Execute the file content with NumPy in scope

channel_groups = local_vars.get('channel_groups')
if channel_groups is None:
    raise ValueError(f"'channel_groups' not found in {file_path}")

# Assuming channel_groups is loaded from Step 1
data = []
for group_id, group_data in channel_groups.items():
    channels = group_data['channels']
    geometry = group_data['geometry']
    for channel in channels:
        x, y = geometry[channel]
        data.append({
            'group_id': group_id,
            'channel_id': channel,
            'x': x,
            'y': y
        })
probe_df = pd.DataFrame(data)

# 你的数据和 DataFrame
data = probe_df  # 我省略了完整数据，因为它已经在上文给出

# 分组函数
def group_channels_by_group(df, channels):
    grouped = {}
    for channel in channels:
        group = df[df['channel_id'] == channel]['group_id'].values[0]
        if group not in grouped:
            grouped[group] = []
        grouped[group].append(channel)
    return grouped

# 找到每个组的中间 channel_id
def find_middle_channel_per_group(df, grouped_channels):
    middle_channels = {}
    for group, channels in grouped_channels.items():
        group_df = df[df['channel_id'].isin(channels)]
        sorted_df = group_df.sort_values('y', ascending=False).reset_index(drop=True)
        middle_idx = len(sorted_df) // 2
        middle_channel = sorted_df.iloc[middle_idx]['channel_id']
        middle_channels[group] = middle_channel
    return middle_channels

# 映射中间 channel 到输入列表
def map_middle_channels_to_input(df, channel_list, middle_channels):
    output_list = []
    for channel in channel_list:
        group = df[df['channel_id'] == channel]['group_id'].values[0]
        output_list.append(middle_channels[group])
    return output_list

# 找到每个组中间 channel 的下第 4 个
def find_lower_four_channel_per_group(df, grouped_channels, middle_channels):
    lower_four_dict = {}
    for group, middle_channel in middle_channels.items():
        group_df = probe_df[probe_df['group_id'] == group][['channel_id', 'y']]
        sorted_group = group_df.sort_values('y', ascending=False).reset_index(drop=True)
        start_idx = sorted_group[sorted_group['channel_id'] == middle_channel].index[0]
        target_idx = start_idx + 4
        if target_idx < len(sorted_group):
            lower_four_dict[group] = sorted_group.iloc[target_idx]['channel_id']
        else:
            lower_four_dict[group] = np.nan
    return lower_four_dict

# 映射下第 4 个到输入列表
def map_lower_four_to_input(df, channel_list, lower_four_dict):
    lower_four_output_list = []
    for channel in channel_list:
        group = df[df['channel_id'] == channel]['group_id'].values[0]
        lower_four_output_list.append(lower_four_dict[group])
    return lower_four_output_list
# Function to get pickle files
def get_pkl_files(folder_path):
    all_files = os.listdir(folder_path)
    pkl_files = [f for f in all_files if f.endswith("withDLC.pkl")]
    return pkl_files

# Define group prefixes
target_prefixes_control = ['65165', '65091', '63383', '66539', '65622']
target_prefixes_exp = ['65588', '63385', '66538', '66537', '66922']

In [ ]:
### Collecting LFPs
base_folder = r"/Volumes/quattrocolo/crhip/Sachuriga/Ephys_Recording/CR_CA1"
pkl_files = get_pkl_files(r"/Users/sachuriga/Desktop/Projects/CR_CA1_paper/file_with_table/ripple_ch")
pkl_fodler = r"/Users/sachuriga/Desktop/Projects/CR_CA1_paper/file_with_table/ripple_ch"
phy_folder = r"/Volumes/quattrocolo/crhip/Sachuriga/Ephys_Recording/CR_CA1"
all_dataframes = []

for file in pkl_files:
    try:
        # Read and process the pickle file
        #df_files = pd.read_pickle(fr"{base_folder}{file}")
        unit_table = pd.read_pickle(fr"{pkl_fodler}/{file}")
        pyramidal_df = unit_table[unit_table['cell_type']=="pyramidal"]
        
        # Check if pyramidal_df is empty
        if pyramidal_df.empty:
            print(f"No pyramidal cells found in {file}")
            continue
        
        file_path_nwb = pyramidal_df['session_id'].iloc[0]
        channel_list = pyramidal_df['ripple_ch_3std'].values

        # Channel grouping and calculations
        grouped_channels = group_channels_by_group(probe_df, channel_list)
        middle_channels = find_middle_channel_per_group(probe_df, grouped_channels)
        output_list = map_middle_channels_to_input(probe_df, channel_list, middle_channels)
        lower_four_dict = find_lower_four_channel_per_group(probe_df, grouped_channels, middle_channels)
        lower_four_output_list = map_lower_four_to_input(probe_df, channel_list, lower_four_dict)

        # Print results
        print(f"\nProcessing file: {file}")
        print("Grouped channels:", grouped_channels)
        print("Middle channel per group:", middle_channels)
        print("Output list (length 10):", output_list)
        print("Lower four dict (by group):", lower_four_dict)
        print("Lower four output list:", lower_four_output_list)

        # Channel selection
        chs_py = np.int64(list(middle_channels.values()))
        chs_sr = np.int64([num for num in lower_four_dict.values() if not np.isnan(num)])


        phy_file = unit_table['session_id'].iloc[0].split(".nwb")[0]
        animals_id = unit_table['session_id'].iloc[0].split("_")[0]
        try:
            bad_channels = np.load(rf"{phy_folder}/{animals_id}/{phy_file}/bad_channels.npy")
        except Exception as e:
            bad_channels = []

        chs_py = [x for x in chs_py if x not in bad_channels]
        chs_sr = [x for x in chs_sr if x not in bad_channels]

        if not chs_py:
            continue
        if not chs_sr:
            continue
        os_fodler = unit_table['session_id'].iloc[0].split("_phy_k_manual.nwb")[0]
        temp_folder = r"C:/temp_lfp"
        animal_id = pyramidal_df['animal_id'].iloc[0]
        session_id = pyramidal_df['session_id'].iloc[0]
        day = pyramidal_df['matlab_day'].iloc[0]
        path = fr"{base_folder }/{animal_id}/{phy_file}/"
        raw_path = fr"{base_folder}/{animal_id}/{os_fodler}/"
        # Load NWB file
        filepath = rf"S:\Sachuriga\nwb\test4neo/{file_path_nwb}"
        npdata = nap.load_file(filepath)
        down_sample_lfp_test(path,raw_path)

        # Extract data
        eeg_raw = np.load(rf"{phy_folder}/{animals_id}/{phy_file}/lfp_zscore.npy")
        lfp_times = np.load(rf"{phy_folder}/{animals_id}/{phy_file}/lfp_times.npy")
        eeg = nap.TsdFrame(t=lfp_times, d=eeg_raw, time_units='s')
        
        forward_ep = npdata['Mid_brain_cords']
        
        if  forward_ep.index.values[-1] >lfp_times[-1]:
            forward_ep = forward_ep[forward_ep.index.values<lfp_times[-1],:]

        RUN_interval = nap.IntervalSet(forward_ep.start, forward_ep.end)
        
        # Position and speed processing
        pos_cord = load_speed_fromNWB(forward_ep)
        raw_pos, combined_array, mask, speeds, smoothed_speed, filtered_speed = pos2speed(
            pos_cord[:,0], pos_cord[:,1], pos_cord[:,2],
            filter_speed=True, min_speed=0.05
        )
        time_stemp = pos_cord[:,0]

        # Initialize lists for this iteration
        lfp_py = []
        lfp_sr = []
        lfp_py_time = []
        lfp_sr_time = []
        
        # Get identifiers
        # Process pyramidal channels
        if any(chs_py):
            for ch_num in chs_py:
                eeg_example = eeg.restrict(RUN_interval)[:, ch_num]
                fs = 1250
                t = eeg_example.t
                lfp_data = eeg_example.d

                # Bandpass filter
                lowcut = 1
                highcut = 400
                nyquist = fs / 2
                order = 4
                b, a = signal.butter(order, [lowcut / nyquist, highcut / nyquist], btype='band')
                bandpassed_data = signal.filtfilt(b, a, lfp_data)
                eeg_example = nap.Tsd(t=t, d=bandpassed_data)

                lfp_py_time.append(t)
                lfp_py.append(eeg_example)

        # Process SR channels
        if any(chs_sr):
            for ch_num in chs_sr:
                eeg_example = eeg.restrict(RUN_interval)[:, ch_num]
                fs = 1250
                t = eeg_example.t
                lfp_data = eeg_example.d

                # Bandpass filter
                lowcut = 1
                highcut = 400
                nyquist = fs / 2
                order = 4
                b, a = signal.butter(order, [lowcut / nyquist, highcut / nyquist], btype='band')
                bandpassed_data = signal.filtfilt(b, a, lfp_data)
                eeg_example = nap.Tsd(t=t, d=bandpassed_data)

                lfp_sr_time.append(t)
                lfp_sr.append(eeg_example)

        # Create DataFrame for this iteration
        iteration_df = pd.DataFrame({
            'animal_id': [animal_id],
            'session_id': [session_id],
            'matlab_day': [day],
            'lfp_py': [lfp_py],
            'lfp_sr': [lfp_sr],
            'lfp_py_time': [lfp_py_time],
            'lfp_sr_time': [lfp_sr_time],
            'smoothed_speed': [smoothed_speed],
            'time_stemp': [time_stemp],
            'file_path': [file]  # Adding source file info
        })
        
        # Append to list of DataFrames
        all_dataframes.append(iteration_df)
    except Exception as e:
        continue

# Combine all DataFrames into one
final_df = pd.concat(all_dataframes, ignore_index=True)

# Optional: Save to file
final_df.to_pickle(r'/Users/sachuriga/Desktop/Projects/CR_CA1_paper/tables/combined_data_for_LFP.pkl')

print(f"\nProcessed {len(all_dataframes)} files")
print(f"Final DataFrame shape: {final_df.shape}")
print("Columns:", final_df.columns.tolist())

In [ ]:
import numpy as np
from scipy.signal import butter, filtfilt, hilbert
from scipy.fft import fft, fftfreq

def butter_bandpass_butter(lowcut, highcut, fs, order=4):
    """Creates a Butterworth bandpass filter."""
    nyquist = 0.5 * fs
    low = lowcut / nyquist
    high = highcut / nyquist
    b, a = butter(order, [low, high], btype='band')
    return b, a

def gamma_event(signal_data, time_input, gamma_band, fs=1250):
    """
    Calculates gamma event rate and theta-gamma phase-amplitude coupling.
    
    Parameters:
    - signal_data: 1D numpy array of LFP data
    - time_input: 1D numpy array of time points in seconds
    - gamma_band: tuple of (low_freq, high_freq) for the gamma band of interest
    - fs: Sampling frequency in Hz (default 1250)
    
    Returns:
    - gamma_event_rate: number of events per second
    - coupling_strength: Mean Resultant Vector Length (MRVL)
    """
    
    # Step 1 & 2: Compute time-varying power in the specified gamma band
    window_size_samples = int(0.1 * fs)  # 100 ms window
    step_size_samples = int(0.01 * fs)   # 10 ms step
    
    power_values = []
    window_centers_samples = [] # Keep track of samples to avoid rounding errors

    for start in range(0, len(signal_data) - window_size_samples, step_size_samples):
        window = signal_data[start:start + window_size_samples]
        f = fftfreq(window_size_samples, 1/fs)
        psd = np.abs(fft(window))**2
        
        # Isolate power in the specific gamma band
        idx = np.logical_and(f >= gamma_band[0], f <= gamma_band[1])
        avg_power = np.mean(psd[idx])
        
        window_centers_samples.append(start + window_size_samples // 2)
        power_values.append(avg_power)

    power_values = np.array(power_values)
    window_centers_samples = np.array(window_centers_samples)

    # Step 3: Threshold power at 2 SD above mean
    mean_power = np.mean(power_values)
    std_power = np.std(power_values)
    threshold = mean_power + 2 * std_power
    
    # Get the sample indices where power exceeds the threshold
    high_power_centers = window_centers_samples[power_values > threshold]

    # Step 4: Filter signal in the specified gamma band
    b_gamma, a_gamma = butter_bandpass_butter(gamma_band[0], gamma_band[1], fs)
    gamma_filtered = filtfilt(b_gamma, a_gamma, signal_data)

    # Step 5: Extract 160 ms windows around high-power points and find maxima
    window_160ms_samples = int(0.160 * fs / 2)  # Half window (80ms) in samples
    raw_maxima_indices = []

    for center_idx in high_power_centers:
        start = max(0, center_idx - window_160ms_samples)
        end = min(len(signal_data), center_idx + window_160ms_samples)
        
        segment = gamma_filtered[start:end]
        if len(segment) > 0:
            peak_idx = start + np.argmax(segment)
            raw_maxima_indices.append(peak_idx)

    # Step 6: Remove duplicates and enforce 100 ms separation using sample indices
    raw_maxima_indices = np.unique(raw_maxima_indices)
    min_separation_samples = int(0.100 * fs)  # 100 ms in samples
    
    filtered_maxima_indices = []
    for i, peak_idx in enumerate(raw_maxima_indices):
        if i == 0 or (peak_idx - filtered_maxima_indices[-1]) >= min_separation_samples:
            filtered_maxima_indices.append(peak_idx)
            
    filtered_maxima_indices = np.array(filtered_maxima_indices)

    # --- Theta-Gamma Coupling Analysis ---

    # Step 7: Bandpass filter LFP in theta range (6-10 Hz)
    b_theta, a_theta = butter_bandpass_butter(6, 10, fs)
    theta_filtered = filtfilt(b_theta, a_theta, signal_data)

    # Step 8: Compute theta phase using Hilbert transform
    analytic_signal = hilbert(theta_filtered)
    theta_phase_rad = np.angle(analytic_signal)  # Phases in radians (-pi to pi)

    # Step 9: Extract theta phases exactly at the gamma maxima indices
    # (This avoids the clunky time-matching search from before)
    valid_indices = filtered_maxima_indices[filtered_maxima_indices < len(theta_phase_rad)]
    gamma_phases_rad = theta_phase_rad[valid_indices]

    # Step 10: Calculate resultant vector (Coupling Strength)
    phase_vectors = np.exp(1j * gamma_phases_rad)
    coupling_strength = np.abs(np.mean(phase_vectors)) 

    # Step 11: Phase Distribution (Fixing the wrapping bug)
    # Convert to degrees and wrap to 0-360 range
    gamma_phases_deg = np.mod(np.degrees(gamma_phases_rad), 360)
    bins = np.arange(0, 360 + 30, 30)
    gamma_phase_hist, _ = np.histogram(gamma_phases_deg, bins=bins)
    
    total_gamma_events = len(valid_indices)
    
    # Avoid division by zero if no events are found
    if total_gamma_events > 0:
        gamma_phase_dist = gamma_phase_hist / total_gamma_events
    else:
        gamma_phase_dist = np.zeros_like(gamma_phase_hist)

    # Calculate gamma events per second
    signal_duration = len(signal_data) / fs
    gamma_event_rate = total_gamma_events / signal_duration

    print(f"--- Results for Gamma Band {gamma_band} Hz ---")
    print(f"Total gamma events: {total_gamma_events}")
    print(f"Signal duration: {signal_duration:.2f} seconds")
    print(f"Gamma event rate: {gamma_event_rate:.2f} events/second")
    print(f"Strength of theta-gamma coupling (MRVL): {coupling_strength:.3f}\n")

    return gamma_event_rate, coupling_strength

# # Example Usage (assuming you have 'signal' and 't' defined)
# slow_gamma_band = (20, 41)
# slow_event_rate, slow_coupling = gamma_event(signal, t, slow_gamma_band)

# fast_gamma_band = (39, 91)
# fast_event_rate, fast_coupling = gamma_event(signal, t, fast_gamma_band)

In [ ]:
import pandas as pd
final_df = pd.read_pickle(r"/Users/sachuriga/Desktop/Projects/CR_CA1_paper/tables/combined_data_for_LFP.pkl")

In [ ]:
#final_df = pd.read_pickle(r"Q:/sachuriga/CR_CA1_paper/tables/combined_data_for_LFP.pkl")
fs=1250
all_temp_rows = []
i=0
for index,row in final_df.iterrows():

    smoothed_speed = row['smoothed_speed']
    time_stemp = row['time_stemp'] 
    ## calculate ran epoch
    starts,stops = find_run_indices(smoothed_speed, threshold=0.05)
    run_ep = nap.IntervalSet(start=time_stemp[starts], end=time_stemp[stops], time_units='s')
    wake_ep = nap.IntervalSet(start=time_stemp[0], end=time_stemp[-1], time_units='s')
    rest_ep = wake_ep.set_diff(run_ep)
    ## Calculated the power
    #power = nap.compute_power_spectral_density(eeg_example, fs=1000, ep=wake_ep)

    if  any(final_df['lfp_sr']):
        lfp_sr_norm = []
        lfp_sr_norm_run = []
        lfp_sr_norm_rest = []
        sr=[]
        sttr=[]
        fsr=[]
        fstr=[]
        for eeg_example in row['lfp_sr']:
            fs=1250
            starts,stops = find_run_indices(smoothed_speed, threshold=0.05)
            run_ep = nap.IntervalSet(start=time_stemp[starts], end=time_stemp[stops], time_units='s')
            wake_ep = nap.IntervalSet(start=time_stemp[0], end=time_stemp[-1], time_units='s')
            rest_ep = wake_ep.set_diff(run_ep)

            power = nap.compute_power_spectral_density(eeg_example, fs=fs, ep=wake_ep)
            # Define the frequency range for normalization (1–100 Hz)

            freqs = power.index.values  # Frequency array
            power_vals = power.values    # Power array

            freq_mask = (freqs >= 1) & (freqs <= 400)
            freqs_range = freqs[freq_mask]
            power_range = power_vals[freq_mask]

            # Compute total power in the 1–100 Hz band
            total_power = np.sum(power_range)

            # Normalize the power
            normalized_power = power_vals / total_power

            power_run = nap.compute_mean_power_spectral_density(
                eeg_example, 1.5, fs=fs, ep=run_ep
            )
            power_rest = nap.compute_mean_power_spectral_density(
                eeg_example, 1.5, fs=fs, ep=rest_ep
            )
            # Normalize run and rest power
            power_run[0] = power_run.values / np.sum(power_run.values[(power_run.index >= 1) & (power_run.index <= 400)])
            power_rest[0]  = power_rest.values / np.sum(power_rest.values[(power_rest.index >= 1) & (power_rest.index <= 400)])
            lfp_sr_norm_run.append(power_run[0])
            lfp_sr_norm_rest.append(power_rest[0])
            fs=1250
            signal = eeg_example.restrict(run_ep).values
            t = eeg_example.restrict(run_ep).index.values
            # Example usage
            # Example usage
            slow_gamma_band = (20,41)  # Define gamma band
            slow_event_rate, slow_theta_gamma_coupling = gamma_event(signal,t, slow_gamma_band)
            fast_gamma_band = (39,91)
            fast_event_rate, fast_theta_gamma_coupling = gamma_event(signal,t, fast_gamma_band)
            sr.append(slow_event_rate)
            sttr.append(slow_theta_gamma_coupling)
            fsr.append(fast_event_rate)
            fstr.append(fast_theta_gamma_coupling )
        row['lfp_sr_norm_run']=lfp_sr_norm_run
        row['lfp_sr_norm_rest']=lfp_sr_norm_rest
        row['slow_event_rate_sr']=sr
        row['slow_theta_gamma_coupling_sr']=sttr
        row['fast_event_rate_sr']=fsr
        row['fast_theta_gamma_coupling_sr']=fstr

    if  any(final_df['lfp_py']):
        lfp_py_norm = []
        lfp_py_norm_run = []
        lfp_py_norm_rest = []
        sr=[]
        sttr=[]
        fsr=[]
        fstr=[]
        for eeg_example in row['lfp_py']:
            fs=1250
            starts,stops = find_run_indices(smoothed_speed, threshold=0.05)
            run_ep = nap.IntervalSet(start=time_stemp[starts], end=time_stemp[stops], time_units='s')
            wake_ep = nap.IntervalSet(start=time_stemp[0], end=time_stemp[-1], time_units='s')
            rest_ep = wake_ep.set_diff(run_ep)

            power = nap.compute_power_spectral_density(eeg_example, fs=fs, ep=wake_ep)
            # Define the frequency range for normalization (1–100 Hz)

            freqs = power.index.values  # Frequency array
            power_vals = power.values    # Power array

            freq_mask = (freqs >= 1) & (freqs <= 400)
            freqs_range = freqs[freq_mask]
            power_range = power_vals[freq_mask]

            # Compute total power in the 1–100 Hz band
            total_power = np.sum(power_range)

            # Normalize the power
            normalized_power = power_vals / total_power

            power_run = nap.compute_mean_power_spectral_density(
                eeg_example, 1.5, fs=fs, ep=run_ep
            )
            power_rest = nap.compute_mean_power_spectral_density(
                eeg_example, 1.5, fs=fs, ep=rest_ep
            )
            # Normalize run and rest power
            power_run[0] = power_run.values / np.sum(power_run.values[(power_run.index >= 1) & (power_run.index <= 400)])
            power_rest[0]  = power_rest.values / np.sum(power_rest.values[(power_rest.index >= 1) & (power_rest.index <= 400)])
            lfp_py_norm_run.append(power_run[0])
            lfp_py_norm_rest.append(power_rest[0])
            slow_gamma_band = (20,41)  # Define gamma band
            slow_event_rate, slow_theta_gamma_coupling = gamma_event(signal,t, slow_gamma_band)
            fast_gamma_band = (39,91)
            fast_event_rate, fast_theta_gamma_coupling = gamma_event(signal,t, fast_gamma_band)
            sr.append(slow_event_rate)
            sttr.append(slow_theta_gamma_coupling)
            fsr.append(fast_event_rate)
            fstr.append(fast_theta_gamma_coupling )
        row['lfp_py_norm_run']=lfp_sr_norm_run
        row['lfp_py_norm_rest']=lfp_sr_norm_rest
        row['slow_event_rate_py']=sr
        row['slow_theta_gamma_coupling_py']=sttr
        row['fast_event_rate_py']=fsr
        row['fast_theta_gamma_coupling_py']=fstr
        
    all_temp_rows.append(pd.DataFrame([row[::-1]], columns=row.index[::-1]))
    # i+=1
    # if i==2:
    #     break
combined_temp_nwb = pd.concat(all_temp_rows, ignore_index=True)
combined_temp_nwb.to_pickle(r"/Users/sachuriga/Desktop/Projects/CR_CA1_paper/tables/lfp_with_gamma_event_coupling.pkl")

In [ ]:
all_temp_rows = []

for index, row in final_df.iterrows():
    # 1. Calculate intervals ONCE per row
    time_stemp = row['time_stemp']
    smoothed_speed = row['smoothed_speed']
    starts, stops = find_run_indices(smoothed_speed, threshold=0.05)
    
    wake_ep = nap.IntervalSet(start=time_stemp[0], end=time_stemp[-1], time_units='s')
    run_ep = nap.IntervalSet(start=time_stemp[starts], end=time_stemp[stops], time_units='s')
    rest_ep = wake_ep.set_diff(run_ep)

    # Helper function to avoid code duplication
    def process_lfp_list(lfp_list, run_ep, rest_ep, wake_ep):
        norm_run, norm_rest = [], []
        s_rate, s_coupling, f_rate, f_coupling = [], [], [], []
        
        for eeg_example in lfp_list:
            # Power Spectral Density
            # Note: We compute mean PSD directly for efficiency
            p_run = nap.compute_mean_power_spectral_density(eeg_example, 1.5, fs=1250, ep=run_ep)
            p_rest = nap.compute_mean_power_spectral_density(eeg_example, 1.5, fs=1250, ep=rest_ep)
            
            # Normalize (1-400 Hz)
            mask_run = (p_run.index >= 1) & (p_run.index <= 400)
            norm_run.append(p_run.values / np.sum(p_run.values[mask_run]))
            
            mask_rest = (p_rest.index >= 1) & (p_rest.index <= 400)
            norm_rest.append(p_rest.values / np.sum(p_rest.values[mask_rest]))
            
            # Gamma Events - IMPORTANT: Update signal/t for this specific electrode
            sig_run = eeg_example.restrict(run_ep)
            signal_vals = sig_run.values
            t_vals = sig_run.index.values
            
            sr, sttr = gamma_event(signal_vals, t_vals, (20, 41))
            fr, ftr = gamma_event(signal_vals, t_vals, (39, 91))
            
            s_rate.append(sr); s_coupling.append(sttr)
            f_rate.append(fr); f_coupling.append(ftr)
            
        return norm_run, norm_rest, s_rate, s_coupling, f_rate, f_coupling

    # Process SR layer
    if isinstance(row.get('lfp_sr'), (list, np.ndarray)):
        res = process_lfp_list(row['lfp_sr'], run_ep, rest_ep, wake_ep)
        row['lfp_sr_norm_run'], row['lfp_sr_norm_rest'], \
        row['slow_event_rate_sr'], row['slow_theta_gamma_coupling_sr'], \
        row['fast_event_rate_sr'], row['fast_theta_gamma_coupling_sr'] = res

    # Process PY layer
    if isinstance(row.get('lfp_py'), (list, np.ndarray)):
        res = process_lfp_list(row['lfp_py'], run_ep, rest_ep, wake_ep)
        row['lfp_py_norm_run'], row['lfp_py_norm_rest'], \
        row['slow_event_rate_py'], row['slow_theta_gamma_coupling_py'], \
        row['fast_event_rate_py'], row['fast_theta_gamma_coupling_py'] = res

    all_temp_rows.append(row.to_frame().T)

combined_temp_nwb = pd.concat(all_temp_rows, ignore_index=True)
combined_temp_nwb.to_pickle(r"/Users/sachuriga/Desktop/Projects/CR_CA1_paper/tables/lfp_with_gamma_event_coupling.pkl")

In [ ]:
combined_temp_nwb.to_pickle(r"/Users/sachuriga/Desktop/Projects/CR_CA1_paper/tables/lfp_with_gamma_event_coupling.pkl")

In [ ]:
import numpy as np
from scipy.signal import butter, filtfilt, welch
from scipy.fft import fft, fftfreq
import numpy as np
from scipy.signal import butter, filtfilt, hilbert
import pynapple as nap
import numpy as np
from scipy.signal import butter, filtfilt, hilbert
from scipy.fft import fft, fftfreq
# Bandpass filter for gamma (30-100 Hz)

def butter_bandpass(lowcut, highcut, fs, order=4):
    nyquist = 0.5 * fs
    low = lowcut / nyquist
    high = highcut / nyquist
    b, a = butter(order, [low, high], btype='band')
    return b, a

def gamma_event(signal, time_input, gamma_band):
    # Simulate a signal (replace with your actual LFP data)
    fs = 1250
    signal = signal
    t =  time_input  # Time in seconds


    # Step 1 & 2: Compute time-varying power in gamma band
    window_size = 0.1 * fs  # 100 ms window
    step_size = 0.01 * fs   # 10 ms step
    times = []
    power_values = []

    for start in range(0, len(signal) - int(window_size), int(step_size)):
        window = signal[start:start + int(window_size)]
        f = fftfreq(int(window_size), 1/fs)
        psd = np.abs(fft(window))**2
        idx = np.logical_and(f >= gamma_band[0], f <= gamma_band[1])
        avg_power = np.mean(psd[idx])
        times.append(start + window_size / 2)  # Center of window
        power_values.append(avg_power)

    times = np.array(times) * (1250 / fs)  # Convert to ms
    power_values = np.array(power_values)

    # Step 3: Threshold power at 2 SD above mean
    mean_power = np.mean(power_values)
    std_power = np.std(power_values)
    threshold = mean_power + 2 * std_power
    high_power_times = times[power_values > threshold]

    # Step 4: Extract 160 ms windows around high-power points
    window_160ms = 160  # ms
    samples_160ms = int(window_160ms * fs / 1250 / 2)  # Half window in samples
    gamma_windows = []

    b, a = butter_bandpass(30, 100, fs)
    filtered_signal = filtfilt(b, a, signal)

    for t in high_power_times:
        center = int(t * fs / 1250)
        start = max(0, center - samples_160ms)
        end = min(len(signal), center + samples_160ms)
        gamma_windows.append((start, end))

    # Step 5: Find maxima in gamma filtered signal within each window
    maxima_times = []
    for start, end in gamma_windows:
        segment = filtered_signal[start:end]
        peak_idx = start + np.argmax(segment)
        maxima_times.append(peak_idx * 1250 / fs)  # Convert to ms

    # Step 6: Remove duplicates and enforce 100 ms separation
    maxima_times = np.unique(maxima_times)  # Remove identical maxima
    filtered_maxima = []
    for i, t in enumerate(maxima_times):
        if i == 0 or (t - maxima_times[i-1]) >= 100:
            filtered_maxima.append(t)

    # Step 7: Construct 400 ms windows around maxima from original signal
    window_400ms = 400  # ms
    samples_400ms = int(window_400ms * fs / 1250 / 2)  # Half window in samples
    final_windows = []

    for t in filtered_maxima:
        center = int(t * fs / 1250)
        start = max(0, center - samples_400ms)
        end = min(len(signal), center + samples_400ms)
        final_windows.append(signal[start:end])

    # Theta-gamma coupling analysis
    data = signal  # Raw NumPy array
    times = time_input  # Time points in seconds

    # Step 1: Bandpass filter LFP in theta range (6-10 Hz)
    b_theta, a_theta = butter_bandpass(6, 10, fs)
    theta_filtered = filtfilt(b_theta, a_theta, data)

    # Step 2: Compute theta phase using Hilbert transform
    analytic_signal = hilbert(theta_filtered)
    theta_phase = np.angle(analytic_signal)  # Phases in radians (-pi to pi)

    # Step 3: Extract theta phases at gamma maxima
    gamma_maxima_idx = [np.argmin(np.abs(times * 1250 - t)) for t in filtered_maxima]
    gamma_phases_rad = theta_phase[gamma_maxima_idx]  # Phases in radians

    # Step 4: Calculate resultant vector from phase distribution
    # Convert phases to complex numbers (unit vectors on the complex plane)
    phase_vectors = np.exp(1j * gamma_phases_rad)
    resultant_vector = np.mean(phase_vectors)  # Mean resultant vector
    coupling_strength = np.abs(resultant_vector)  # Length of resultant vector

    # Step 5: Bin gamma phases into 30° bins for distribution (optional visualization)
    theta_phase_deg = np.degrees(theta_phase)  # Convert to degrees for binning
    gamma_phases_deg = theta_phase_deg[gamma_maxima_idx]
    bins = np.arange(0, 360 + 30, 30)  # 12 bins: [0-30), [30-60), ..., [330-360)
    gamma_phase_hist, _ = np.histogram(gamma_phases_deg, bins=bins)

    # Step 6: Normalize by total number of gamma events
    total_gamma_events = len(filtered_maxima)
    gamma_phase_dist = gamma_phase_hist / total_gamma_events

    # Step 7: Calculate gamma events per second
    signal_duration = len(times)/1250  # Duration in seconds
    gamma_event_rate = total_gamma_events / signal_duration

    print(f"\nTotal gamma events: {total_gamma_events}")
    print(f"Signal duration: {signal_duration:.2f} seconds")
    print(f"Gamma event rate: {gamma_event_rate:.2f} events/second")
    print(f"Strength of theta-gamma coupling (resultant vector length): {coupling_strength:.3f}")

    return gamma_event_rate, coupling_strength

# Example usage
slow_gamma_band = (20,41)  # Define gamma band
slow_event_rate, slow_theta_gamma_coupling = gamma_event(signal,t, slow_gamma_band)
fast_gamma_band = (39,91)
fast_event_rate, fast_theta_gamma_coupling = gamma_event(signal,t, fast_gamma_band)

In [ ]:
fig, ax = plt.subplots(1, constrained_layout=True, figsize=(10, 4))
ax.plot(
    power_run[(power_run.index >= 1.0) & (power_run.index <= 100)],
    alpha=1,
    label="Run",
    linewidth=2,
)
ax.plot(
    power_rest[(power_rest.index >= 1.0) & (power_rest.index <= 80)],
    alpha=1,
    label="Rest",
    linewidth=2,
)
ax.axvspan(6, 12, color="red", alpha=0.1)
ax.set_xlabel("Freq (Hz)")
ax.set_ylabel("Power/Frequency")
ax.set_title("LFP Fourier Decomposition")
ax.legend()

In [ ]:
power_run[(power_run.index >= 1.0) & (power_run.index <= 100)]

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal

def plot_cross_freq_coherence(freq_power, freqs=None, fs=1250, window_size=None, 
                             vmin=None, vmax=None, cmap='viridis'):
    """
    Plot cross-frequency coherence from a power spectrum vector with simulated coupling.
    
    Parameters:
    -----------
    freq_power : array-like
        Vector where values are power and indices correspond to frequencies
    freqs : array-like, optional
        Actual frequency values corresponding to indices. If None, uses indices
    fs : float, optional
        Sampling frequency (default: 1250 Hz)
    window_size : int, optional
        Size of the window for coherence calculation (default: auto-set)
    vmin : float, optional
        Minimum value for color scale (default: auto-set)
    vmax : float, optional
        Maximum value for color scale (default: auto-set)
    cmap : str, optional
        Colormap to use (default: 'viridis')
    
    Returns:
    -------
    fig : matplotlib.figure.Figure
        The generated figure object
    """
    
    # Convert input to numpy array and ensure 1D
    power_data = np.array(freq_power).flatten()
    if freqs is None:
        freqs = np.arange(len(power_data))
    else:
        freqs = np.array(freqs).flatten()
    
    if len(freqs) != len(power_data):
        raise ValueError(f"Length of freqs ({len(freqs)}) must match length of freq_power ({len(power_data)})")
    
    # Create two synthetic time series with some coupling
    t = np.linspace(0, 1, fs)
    signal1 = np.zeros(len(t))
    signal2 = np.zeros(len(t))
    np.random.seed(42)  # For reproducibility
    for i, freq in enumerate(freqs):
        phase_shift = np.random.uniform(0, np.pi)  # Random phase shift for coupling
        amplitude = np.sqrt(max(power_data[i], 0))
        signal1 += amplitude * np.sin(2 * np.pi * freq * t)
        signal2 += amplitude * np.sin(2 * np.pi * freq * t + phase_shift)  # Coupled signal
    
    # Auto-set window size
    if window_size is None:
        window_size = min(256, len(signal1) // 2)
    
    # Calculate coherence between the two signals
    f, Cxy = signal.coherence(signal1, signal2, fs=fs, nperseg=window_size)
    
    # Create coherence matrix
    n_freqs = len(freqs)
    coherence_matrix = np.zeros((n_freqs, n_freqs))
    for i in range(n_freqs):
        for j in range(n_freqs):
            idx_i = np.argmin(np.abs(f - freqs[i]))
            idx_j = np.argmin(np.abs(f - freqs[j]))
            coherence_matrix[i, j] = Cxy[idx_i] if idx_i == idx_j else np.mean(Cxy[[idx_i, idx_j]])
    
    # Auto-adjust vmin and vmax
    if vmin is None:
        vmin = np.min(coherence_matrix)
    if vmax is None:
        vmax = np.max(coherence_matrix)
    
    # Diagnostic output
    print(f"Coherence range: {vmin:.4f} to {vmax:.4f}")
    print(f"Frequency range: {freqs[0]:.2f} to {freqs[-1]:.2f} Hz")
    print(f"Window size: {window_size}")
    
    # Create the plot
    fig, ax = plt.subplots(figsize=(10, 8))
    im = ax.imshow(coherence_matrix, 
                   cmap=cmap,
                   vmin=vmin,
                   vmax=vmax,
                   origin='lower',
                   extent=[freqs[0], freqs[-1], freqs[0], freqs[-1]],
                   aspect='auto')
    
    # Add colorbar
    plt.colorbar(im, label='Coherence')
    
    # Set labels
    ax.set_xlabel('Frequency (Hz)')
    ax.set_ylabel('Frequency (Hz)')
    ax.set_title('Cross-Frequency Coherence')
    
    ax.grid(False)
    plt.tight_layout()
    return fig

# Assuming power_run is a pandas Series with frequency as index and power as values
frequencies = np.array(power_run[(power_run.index >= 1.0) & (power_run.index <= 100)].index.to_numpy())  # First column: frequency
power_spectrum = [num[0] for num in np.array(power_run[(power_run.index >= 1.0) & (power_run.index <= 100)].to_numpy())]  # Second column: power

# Plot
fig = plot_cross_freq_coherence(power_spectrum, freqs=frequencies, fs=1000)
plt.show()

In [ ]:
import numpy as np
from scipy import signal
import matplotlib.pyplot as plt

# Simulated LFP signal with 50 Hz noise
fs = 1000  # Sampling frequency in Hz
t = np.arange(0, 1, 1/fs)  # 1-second time vector
lfp_clean = np.sin(2 * np.pi * 10 * t)  # Simulated 10 Hz LFP signal
noise = 0.5 * np.sin(2 * np.pi * 50 * t)  # 50 Hz noise
lfp_signal = lfp_clean + noise  # Noisy signal

# Design the notch filter
f0 = 50  # Frequency to remove (Hz)
Q = 1   # Quality factor (higher = narrower bandwidth)
b, a = signal.iirnotch(f0 / (fs / 2), Q)  # Normalized frequency: f0 / (fs/2)

# Apply the filter
lfp_filtered = signal.filtfilt(b, a, lfp_signal)  # Zero-phase filtering

# Plot the results
plt.figure(figsize=(10, 6))
plt.subplot(2, 1, 1)
plt.plot(t, lfp_signal, label="Noisy LFP")
plt.plot(t, lfp_filtered, label="Filtered LFP")
plt.title("Time Domain")
plt.legend()

plt.subplot(2, 1, 2)
freq, psd_noisy = signal.welch(lfp_signal, fs, nperseg=1024)
freq, psd_filtered = signal.welch(lfp_filtered, fs, nperseg=1024)
plt.semilogy(freq, psd_noisy, label="Noisy PSD")
plt.semilogy(freq, psd_filtered, label="Filtered PSD")
plt.title("Power Spectral Density")
plt.xlabel("Frequency (Hz)")
plt.legend()
plt.tight_layout()
plt.show()